In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model



# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase2"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:
from huggingface_hub import hf_hub_download

TRAIN_FILES = [
    "Phase_4_CJPE/Train/train.json",
]

train_file_paths = {}

for filename in TRAIN_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    train_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Train/train.json


In [5]:
from huggingface_hub import hf_hub_download

VAL_FILES = [
    "Phase_4_CJPE/Train/dev.json",
]

val_file_paths = {}

for filename in VAL_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    val_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Train/dev.json


In [6]:
from huggingface_hub import hf_hub_download

TEST_FILES = [
    "Phase_4_CJPE/Test/test.json",
    
]

test_file_paths = {}

for filename in TEST_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    test_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE/Test/test.json


In [7]:
import json
import pandas as pd

with open(train_file_paths["Phase_4_CJPE/Train/train.json"], "r") as f:
    train_data = json.load(f)

with open(val_file_paths["Phase_4_CJPE/Train/dev.json"], "r") as f:
    val_data = json.load(f)
with open(test_file_paths["Phase_4_CJPE/Test/test.json"], "r") as f:
    test_data = json.load(f)
    

In [8]:
len(train_data[1000]["text"])

12731

In [9]:
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
train_df = train_df.drop_duplicates().reset_index(drop=True)
val_df = val_df.drop_duplicates().reset_index(drop=True)

# Convert back to lists of dictionaries
train_data = train_df.to_dict(orient="records")
val_data = val_df.to_dict(orient="records")

print("Train records after deduplication:", len(train_data))
print("Validation records after deduplication:", len(val_data))
print("Test records (unchanged):", len(test_data))

Train records after deduplication: 32305
Validation records after deduplication: 2511
Test records (unchanged): 1517


In [10]:
import json

with open("train_deduplicated.json", "w") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open("dev_deduplicated.json", "w") as f:
    json.dump(val_data, f, ensure_ascii=False)

print("Saved cleaned datasets!")

Saved cleaned datasets!


In [11]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [12]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/rogue2222/tokenizer-results/project_bpe_16k.pkl"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    tokenizer = pickle.load(f)
print("BPE vocab size:", len(tokenizer.itos))


BPE vocab size: 7999


In [13]:
converter = ToTokensConverter(tokenizer.itos)

In [14]:
import json
import torch
from torch.utils.data import Dataset, DataLoader

MAX_CHUNK_LEN = 1024
CHUNK_OVERLAP = 64
MAX_CHUNKS = 32


class CJPDataset(Dataset):
    def __init__(
        self,
        json_path,
        converter = converter,
        max_chunk_len=MAX_CHUNK_LEN,
        overlap=CHUNK_OVERLAP,
        max_chunks=MAX_CHUNKS,
        has_labels=True,
    ):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.converter = converter
        self.max_chunk_len = max_chunk_len
        self.overlap = overlap
        self.max_chunks = max_chunks
        self.has_labels = has_labels
        
        assert 0 <= overlap < max_chunk_len
        assert max_chunk_len >= 3, "Need room for <s>, content, and </s>"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Your custom tokenizer should return a list of token IDs.
        token_ids = self.converter.to_tokens([item["text"]])[0]

        # IDs from your itos vocabulary.
        pad_id = 0
        unk_id = 1
        bos_id = 2  # <s>
        eos_id = 3  # </s>

        # Reserve 2 positions for <s> and </s>.
        payload_len = self.max_chunk_len - 2
        step = payload_len - self.overlap

        chunks = []

        for start in range(0, len(token_ids), step):
            content = token_ids[start:start + payload_len]

            if not content:
                break

            chunk_ids = [bos_id] + content + [eos_id]
            chunks.append(chunk_ids)

            if start + payload_len >= len(token_ids):
                break

            if len(chunks) >= self.max_chunks:
                break

        # Handle an empty document.
        if not chunks:
            chunks = [[bos_id, eos_id]]

        return {
            "id": item["id"],
            "chunks": chunks,
            "label": int(item["label"]) if self.has_labels else -1,
        }

In [15]:
def collate_cjp(batch):
    batch_size = len(batch)
    max_chunks = max(len(item["chunks"]) for item in batch)

    max_len = max(
        1,
        max(len(chunk) for item in batch for chunk in item["chunks"])
    )

    pad_id = 0  # <pad>

    input_ids = torch.full(
        (batch_size, max_chunks, max_len),
        pad_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros_like(input_ids)

    chunk_mask = torch.zeros(
        batch_size, max_chunks, dtype=torch.bool
    )

    for i, item in enumerate(batch):
        for j, chunk in enumerate(item["chunks"]):
            n = len(chunk)

            input_ids[i, j, :n] = torch.tensor(
                chunk, dtype=torch.long
            )

            attention_mask[i, j, :n] = 1
            chunk_mask[i, j] = True

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    return {
        "ids": [item["id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "chunk_mask": chunk_mask,
        "labels": labels,
    }

In [16]:
import torch
import torch.nn as nn

In [17]:
import torch
from torch.utils.data import DataLoader

train_dataset = CJPDataset(
    "/kaggle/working/train_deduplicated.json",
    converter=converter,
    has_labels=True,
)

dev_dataset = CJPDataset(
    "/kaggle/working/dev_deduplicated.json",
     converter=converter,
    has_labels=True,
)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_cjp,
)

val_loader = DataLoader(
    dev_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_cjp,
)

In [18]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [19]:

# TODO 3.2: integrate RoPE through attention and the decoder architecture in this section.
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor,attn_mask=None) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim = 1)
            v = v.repeat_interleave(self.n_rep, dim = 1)
       
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.attn_dropout.p if self.training else 0.0
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out



In [20]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE


def build_attn_mask(is_causal, padding_mask, T):
    """is_causal (B,) bool; padding_mask (B,T) bool, True = padding, or None -> (B,1,T,T) bool."""
    full = torch.ones(T, T, dtype=torch.bool, device=is_causal.device)
    allow = torch.where(is_causal[:, None, None, None], torch.tril(full), full)
    if padding_mask is not None:
        allow = allow & ~padding_mask[:, None, None, :].to(torch.bool)
    return allow
    
class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )
        
        self.lm_head.weight = self.tok_emb.weight
                

    def forward(self, idx, padding_mask=None, is_causal=None):                 # CHANGED: is_causal
            B, T = idx.shape
            if padding_mask is not None:
                if padding_mask.shape != (B, T):
                    raise ValueError("padding_mask must have shape (B, T)")
                padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
            if is_causal is None:                                                  # old behaviour: bidirectional
                is_causal = torch.zeros(B, dtype=torch.bool, device=idx.device)
            attn_mask = build_attn_mask(is_causal, padding_mask, T)
            x = self.tok_emb(idx)
            if self.cfg.pos_emb_type == "sinusoidal":
                x = x + self.pos_emb[:T].unsqueeze(0)
            for block in self.blocks:
                x = block(x, attn_mask=attn_mask)
            x = self.ln_f(x)
            return self.lm_head(x) 


In [21]:
class LoraLinear(nn.Module):
    def __init__(self, layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()

        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout)

        # Freeze original layer
        for p in self.layer.parameters():
            p.requires_grad = False

        self.in_features = layer.in_features
        self.out_features = layer.out_features

        # A: [rank, in_features]
        self.A = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        # B: [out_features, rank]
        self.B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5
        )

    def forward(self, x):

        # Original path: W0 x
        W0x = self.layer(x)

        # LoRA path: BAx
        Ax = F.linear(self.dropout(x), self.A)
        BAx = F.linear( Ax, self.B)

        return W0x + (self.alpha / self.rank) * BAx

In [22]:

class LoraModel(nn.Module):
    def __init__(
        self,
        base_model,
        d_model,
        num_classes,
        rank=8,
        alpha=16,
        lora_dropout=0.05,
        dropout=0.2,
        use_pos=False,
        projections=("q_proj", "k_proj"),
    ):
        super().__init__()

        self.model = base_model
        self.use_pos = use_pos
        self.drop = nn.Dropout(dropout)

        # Freeze original Phase-1 parameters.
        for param in self.model.parameters():
            param.requires_grad = False

        # Insert trainable LoRA adapters.
        self._add_lora(
            rank=rank,
            alpha=alpha,
            dropout=lora_dropout,
            projections=projections,
        )

        # Same classification head as RRClassifier.
        self.head = nn.Linear(
            d_model + int(use_pos),
            num_classes,
        )

    def _add_lora(self, rank, alpha, dropout, projections):
        for module in self.model.modules():
            for proj_name in projections:

                # Feed-forward layers, if requested.
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2

                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoraLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            )

                # Attention projections.
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)

                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoraLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            ),
                        )
    def get_hidden_states(self, input_ids, padding_mask):
        B, T = input_ids.shape
    
        x = self.model.tok_emb(input_ids)
    
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
    
        padding_mask = padding_mask.bool()
        is_causal = torch.zeros(
            B, dtype=torch.bool, device=input_ids.device
        )
    
        attn_mask = build_attn_mask(
            is_causal, padding_mask, T
        )
    
        for block in self.model.blocks:
            x = block(x, attn_mask=attn_mask)
    
        x = self.model.ln_f(x)
    
        return x  # Shape: [B, T, d_model]
    def encode(self, input_ids, padding_mask):
        hidden = self.get_hidden_states(input_ids, padding_mask)
    
        if hidden is None:
            raise RuntimeError(
                "get_hidden_states() returned None. "
                "Check its return statement."
            )
    
        valid = (~padding_mask.bool()).unsqueeze(-1).to(hidden.dtype)
        counts = valid.sum(dim=1)
    
        pooled = (hidden * valid).sum(dim=1) / counts.clamp(min=1)
    
        return pooled


    def forward(self, input_ids, padding_mask):
        pooled = self.encode(
            input_ids=input_ids,
            padding_mask=padding_mask,
        )
    
        return self.head(self.drop(pooled))
                    
        


In [23]:
class HierarchicalLoraCJP(nn.Module):
    def __init__(
        self,
        lora_classifier,
        gru_hidden=256,
        dropout=0.2,
        freeze_lora_classifier=True,
    ):
        super().__init__()

        self.chunk_encoder = lora_classifier
        self.freeze_lora_classifier = freeze_lora_classifier

        d_model = self.chunk_encoder.model.cfg.d_model
        chunk_dim = d_model + int(self.chunk_encoder.use_pos)

        # The chunk encoder's original classification head is unused here.
        self.bigru = nn.GRU(
            input_size=chunk_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * gru_hidden, gru_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden, 2),
        )

        if freeze_lora_classifier:
            for p in self.chunk_encoder.parameters():
                p.requires_grad = False

            # Re-enable LoRA adapter parameters.
            for module in self.chunk_encoder.modules():
                if isinstance(module, LoraLinear):
                    for p in module.parameters():
                        p.requires_grad = True

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_lora_classifier:
            self.chunk_encoder.eval()
        return self

    # def forward(self, input_ids, attention_mask, chunk_mask):
    #     # input_ids, attention_mask: [B, K, T]
    #     # chunk_mask: [B, K]
    #     B, K, T = input_ids.shape
    
    #     flat_ids = input_ids.reshape(B * K, T)
    #     flat_attention = attention_mask.reshape(B * K, T).bool()
    
    #     valid_chunks = flat_attention.any(dim=1)
    
    #     chunk_dim = self.chunk_encoder.model.cfg.d_model
    
    #     chunk_vectors = self.chunk_encoder.model.tok_emb.weight.new_zeros(
    #         B * K, chunk_dim
    #     )

    #     if valid_chunks.any():
    #         ids = flat_ids[valid_chunks]
    #         padding_mask = ~flat_attention[valid_chunks]
    
    #         if self.freeze_lora_classifier:
    #             with torch.no_grad():
    #                 vectors = self.chunk_encoder.encode(
    #                     input_ids=ids,
    #                     padding_mask=padding_mask,
    #                 )
    #         else:
                # vectors = self.chunk_encoder.encode(
                #     input_ids=ids,
                #     padding_mask=padding_mask,
                # )
    
        #     chunk_vectors[valid_chunks] = vectors.to(chunk_vectors.dtype)
    
        # chunk_vectors = chunk_vectors.reshape(B, K, chunk_dim)
    
        # lengths = chunk_mask.sum(dim=1).long()
    
        # if (lengths <= 0).any():
        #     raise ValueError("Each document must contain at least one chunk.")
    
        # packed = nn.utils.rnn.pack_padded_sequence(
        #     chunk_vectors,
        #     lengths.cpu(),
        #     batch_first=True,
        #     enforce_sorted=False,
        # )
    
        # _, hidden = self.bigru(packed)
    
        # document_vector = torch.cat(
        #     [hidden[-2], hidden[-1]],
        #     dim=-1,
        # )
    
        # return self.classifier(document_vector)
        
    def forward(
        self,
        input_ids,
        attention_mask,
        chunk_mask,
        chunk_batch_size=2,
    ):
        # input_ids, attention_mask: [B, K, T]
        # chunk_mask: [B, K]
        B, K, T = input_ids.shape
        D = self.chunk_encoder.model.cfg.d_model

        flat_ids = input_ids.reshape(B * K, T)
        flat_attention = attention_mask.reshape(B * K, T).bool()
    
        valid_indices = flat_attention.any(dim=1).nonzero(
            as_tuple=True
        )[0]
    
        # Encode valid chunks in small batches.
        # Keep the output tensors connected to autograd.
        encoded_chunks = []
    
        for start in range(0, len(valid_indices), chunk_batch_size):
            indices = valid_indices[start:start + chunk_batch_size]
    
            ids = flat_ids.index_select(0, indices)
            attention = flat_attention.index_select(0, indices)
            padding_mask = ~attention

            if self.freeze_lora_classifier:
                with torch.no_grad():
                    vectors = self.chunk_encoder.encode(
                        input_ids=ids,
                        padding_mask=padding_mask,
                    )
            else:
                vectors = self.chunk_encoder.encode(
                    input_ids=ids,
                    padding_mask=padding_mask,
                )
    
            encoded_chunks.append((indices, vectors))
    
        # Restore original chunk order without in-place assignment
        # into a detached tensor.
        zero_vector = self.chunk_encoder.model.tok_emb.weight.new_zeros(D)
        output = [zero_vector] * (B * K)
    
        for indices, vectors in encoded_chunks:
            for j, idx in enumerate(indices.tolist()):
                output[idx] = vectors[j]
    
        chunk_vectors = torch.stack(output, dim=0)
        chunk_vectors = chunk_vectors.reshape(B, K, D)
    
        lengths = chunk_mask.sum(dim=1).long()
    
        if (lengths <= 0).any():
            raise ValueError("Each document must contain at least one chunk.")
    
        packed = nn.utils.rnn.pack_padded_sequence(
            chunk_vectors,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
    
        _, hidden = self.bigru(packed)
    
        document_vector = torch.cat(
            [hidden[-2], hidden[-1]],
            dim=-1,
        )
    
        return self.classifier(document_vector)


In [24]:
class LoraPartialFinetuneModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, projections = ("q_proj","k_proj"), unfreeze_last_n = 0):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        if unfreeze_last_n > 0:
            if unfreeze_last_n > len(self.model.blocks):
                raise ValueError("unfreeze_last_n exceeds number of blocks")

            for block in self.model.blocks[-unfreeze_last_n:]:
                for param in block.parameters():
                    param.requires_grad = True
        self._add_lora(rank, alpha, dropout)
        self.ner_head = nn.Linear(d_model, labels)

    def _add_lora(self, rank, alpha, dropout):
        for module in self.model.modules():
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoraLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoraLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            ),
                        )
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        # Transformer blocks
        for block in self.model.blocks:
            x = block(
                x,
                attn_mask=attn_mask
            )
        x = self.model.ln_f(x)

        return x
    def forward(self, x, padding_mask=None):
          hidden_states = self.get_hidden_states(x, padding_mask)
          logits = self.ner_head(hidden_states)
          return logits


In [25]:
class LoRMALinear(nn.Module):
    def __init__(self, layer, rank=16, alpha=32, dropout=0.05, variant = "plus"):
        super().__init__()
        if variant not in {"plus", "pi"}:
            raise ValueError("Only phi or plus supported")
        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.variant = variant
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad = False
        d = layer.out_features
        self.A = nn.Parameter(torch.empty(rank, d))
        self.B = nn.Parameter(torch.empty(d, rank))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=5**0.5)
            nn.init.zeros_(self.B)
        else:
            nn.init.normal_(self.B)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_()
                self.A[0, 0] = 1.0
    @staticmethod
    def inflate_pi(BA):
        """Cyclically rotate row i by i positions."""
        d = BA.shape[0]
        return torch.stack(
            [torch.roll(BA[i], shifts=i, dims=0) for i in range(d)],
            dim=0,
        )
    def forward(self, x):
        base = self.layer(x)

        # Apply BA to the pretrained output
        z = self.dropout(base)
        if self.variant == "plus":
            Az = F.linear(z, self.A)
            BAz = F.linear(Az, self.B)
    
            # h = (I + alpha/rank * BA) W0 x
            output = base + (self.alpha / self.rank) * BAz
            
    
            return output
        else:
            BA = self.B @ self.A

            # Permutation-based rank inflation
            BA_pi = self.inflate_pi(BA)
    
            # h = (I_pi(BA) @ W0) x
            # Equivalent to applying BA_pi to W0 x
            adapted = F.linear(z, BA_pi)
    
            return base + (self.alpha / self.rank) * adapted

In [26]:
class LormaModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, variant="plus", projections=("q_proj", "k_proj")):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        self.variant = variant
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        self._add_lorma(rank, alpha, dropout)
        self.ner_head = nn.Linear(d_model, labels)


    def _add_lorma(self, rank, alpha, dropout):
        for module in self.model.modules():
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoRMALinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoRMALinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            ),
                        )
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        # Transformer blocks
        for block in self.model.blocks:
            x = block(
                x,
                attn_mask=attn_mask
            )
        x = self.model.ln_f(x)

        return x
    def forward(self, x, padding_mask=None):
          hidden_states = self.get_hidden_states(x, padding_mask)
          logits = self.ner_head(hidden_states)
          return logits


In [27]:
import torch
import torch.nn as nn


class HierarchicalCJP(nn.Module):
    def __init__(
        self,
        encoder,
        hidden_dim,
        gru_hidden=256,
        dropout=0.2,
        freeze_encoder=False,
    ):
        super().__init__()

        self.encoder = encoder
        self.freeze_encoder = freeze_encoder

        assert hidden_dim == encoder.cfg.d_model, (
            f"hidden_dim={hidden_dim}, "
            f"but encoder d_model={encoder.cfg.d_model}"
        )

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        self.bigru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * gru_hidden, gru_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden, 2),
        )

    def train(self, mode=True):
        super().train(mode)

        # Keep the frozen encoder's dropout disabled.
        if self.freeze_encoder:
            self.encoder.eval()

        return self

    def get_hidden_states(self, idx, padding_mask=None):
        """
        idx:         [N, T]
        padding_mask:[N, T], True means padding

        Returns:
            hidden states [N, T, d_model]
        """
        B, T = idx.shape
        cfg = self.encoder.cfg

        if T > cfg.max_seq_len:
            raise ValueError(
                f"Chunk length {T} exceeds encoder maximum "
                f"sequence length {cfg.max_seq_len}"
            )

        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )

            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )

        # Token embeddings
        x = self.encoder.tok_emb(idx)

        # Add positional embeddings only for sinusoidal encoding.
        if cfg.pos_emb_type == "sinusoidal":
            x = x + self.encoder.pos_emb[:T].unsqueeze(0)

        # Your encoder is bidirectional.
        is_causal = torch.zeros(
            B, dtype=torch.bool, device=idx.device
        )

        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T,
        )

        for block in self.encoder.blocks:
            x = block(x, attn_mask=attn_mask)

        return self.encoder.ln_f(x)

    def encode_chunks(self, input_ids, attention_mask):
        """
        input_ids:      [B, K, T]
        attention_mask: [B, K, T]

        Returns:
            chunk_vectors: [B, K, d_model]
        """
        B, K, T = input_ids.shape

        flat_ids = input_ids.reshape(B * K, T)
        flat_mask = attention_mask.reshape(B * K, T)

        valid = flat_mask.bool().any(dim=1)

        chunk_vectors = torch.zeros(
            B * K,
            self.encoder.cfg.d_model,
            device=input_ids.device,
            dtype=self.encoder.tok_emb.weight.dtype,
        )

        if valid.any():
            ids = flat_ids[valid]
            mask = flat_mask[valid]

            # True means padding in your custom encoder.
            padding_mask = ~mask.bool()

            if self.freeze_encoder:
                with torch.no_grad():
                    token_states = self.get_hidden_states(
                        idx=ids,
                        padding_mask=padding_mask,
                    )
            else:
                token_states = self.get_hidden_states(
                    idx=ids,
                    padding_mask=padding_mask,
                )

            # Masked mean pooling over valid token positions.
            mask_float = mask.unsqueeze(-1).to(
                token_states.dtype
            )

            pooled = (
                (token_states * mask_float).sum(dim=1)
                / mask_float.sum(dim=1).clamp(min=1)
            )

            chunk_vectors[valid] = pooled.to(
                chunk_vectors.dtype
            )

        return chunk_vectors.reshape(B, K, -1)

    def forward(self, input_ids, attention_mask, chunk_mask):
        chunk_vectors = self.encode_chunks(
            input_ids,
            attention_mask,
        )

        # Number of real chunks per document.
        lengths = chunk_mask.sum(dim=1)

        if (lengths == 0).any():
            raise ValueError(
                "Every document must contain at least one valid chunk"
            )

        packed = nn.utils.rnn.pack_padded_sequence(
            chunk_vectors,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.bigru(packed)

        # [B, gru_hidden] for each direction.
        forward_state = hidden[-2]
        backward_state = hidden[-1]

        doc_vector = torch.cat(
            [forward_state, backward_state],
            dim=-1,
        )

        # [B, 2] classification logits.
        return self.classifier(doc_vector)

In [28]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=10,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model_phase1 = EncoderLM(cfg)

In [29]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model_phase1))

total parameters :  56592384


In [30]:
checkpoint = torch.load(
    "/kaggle/input/datasets/rogue2222/hybrid1b/hybrid_encoder.pt",
    map_location=device,
    weights_only=False
)
state_dict = checkpoint["model_state_dict"]

new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    new_state_dict[key] = value

model_phase1.load_state_dict(new_state_dict)

<All keys matched successfully>

In [31]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Chunk mask:", batch["chunk_mask"].shape)
print("Labels:", batch["labels"].shape)
print("Document IDs:", batch["ids"])

Input IDs: torch.Size([2, 1, 879])
Attention mask: torch.Size([2, 1, 879])
Chunk mask: torch.Size([2, 1])
Labels: torch.Size([2])
Document IDs: ['2009_1040', '1972_48']


In [32]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lora_classifier = LoraModel(
    base_model=model_phase1,
    d_model=model_phase1.cfg.d_model,
    num_classes=2,
    rank=16,
    alpha=32,
    lora_dropout=0.05,
    dropout=0.2,
    use_pos=False,
    projections=("q_proj", "k_proj","v_proj","o_proj"),
)

model = HierarchicalLoraCJP(
    lora_classifier=lora_classifier,
    gru_hidden=256,
    dropout=0.2,
    freeze_lora_classifier=False,  # Enable LoRA adapter training
).to(device)

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# print("Device:", device)
# print("Number of GPUs:", torch.cuda.device_count())

In [33]:
# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# print("GPUs available:", torch.cuda.device_count())
# print("Model device:", device)

In [36]:
import torch
from tqdm.auto import tqdm


def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress = tqdm(train_loader, desc="Training", leave=False)

    for batch in progress:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(
            device, non_blocking=True
        )
        chunk_mask = batch["chunk_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(
            device, dtype=torch.long, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        # FP16 mixed precision on CUDA, without GradScaler
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )
            loss = criterion(logits, labels)

        # Normal backward pass
        loss.backward()

        # Check gradients before clipping
        bad_grads = [
            name for name, p in model.named_parameters()
            if p.requires_grad
            and p.grad is not None
            and not torch.isfinite(p.grad).all()
        ]

        if bad_grads:
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError(
                f"Non-finite gradients detected: {bad_grads[:20]}"
            )

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
            error_if_nonfinite=True,
        )

        optimizer.step()

        # Check parameters after the optimizer step
        bad_params = [
            name for name, p in model.named_parameters()
            if p.requires_grad and not torch.isfinite(p).all()
        ]

        if bad_params:
            raise RuntimeError(
                f"Non-finite parameters after optimizer step: "
                f"{bad_params[:20]}"
            )

        batch_size = labels.size(0)
        predictions = logits.detach().argmax(dim=1)

        total_loss += loss.detach().item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / total_samples:.4f}",
            accuracy=f"{total_correct / total_samples:.4f}",
        )

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
    }


@torch.no_grad()
def evaluate(model, val_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_predictions = []
    all_labels = []

    for batch in tqdm(val_loader, desc="Validation", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(
            device, non_blocking=True
        )
        chunk_mask = batch["chunk_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(
            device, dtype=torch.long, non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )
            loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
        "predictions": all_predictions,
        "labels": all_labels,
    }

In [37]:
# from torch.utils.data import DataLoader

# val_dataset = CJPDataset(
#     json_path=val_file_paths["Phase_4_CJPE/Train/dev.json"],
#     converter=converter,
#     max_chunk_len=MAX_CHUNK_LEN,
#     overlap=CHUNK_OVERLAP,
#     max_chunks=MAX_CHUNKS,
#     has_labels=True,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size = 2,  # Reduce if GPU memory is insufficient
#     shuffle=False,
#     num_workers=2,
#     pin_memory=True,
#     collate_fn=collate_cjp,
# )

In [38]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


checkpoint_path = (
    "/kaggle/input/datasets/rogue2222/p4-hybrid-results/best_hierarchical_lora_cjp_3.pt"
)

model.load_state_dict(
    torch.load(checkpoint_path, map_location="cpu", weights_only=True)
)
model = model.to(device)

model = (
    nn.DataParallel(model)
    if device.type == "cuda" and torch.cuda.device_count() > 1
    else model
)

base = model.module if isinstance(model, nn.DataParallel) else model
# Collect LoRA parameters
lora_params = []
for module in base.modules():
    if isinstance(module, LoraLinear):
        lora_params.extend([module.A, module.B])

lora_ids = {id(p) for p in lora_params}

# Collect trainable BiGRU and classifier parameters
head_params = [
    p for name, p in base.named_parameters()
    if p.requires_grad
    and id(p) not in lora_ids
    and (name.startswith("bigru.") or name.startswith("classifier."))
]

assert lora_params, "No LoRA parameters found"
assert head_params, "No trainable head parameters found"
optimizer = torch.optim.AdamW(
    [
        {"params": lora_params, "lr": 5e-5},
        {"params": head_params, "lr": 3e-4},
    ],
    weight_decay=0.01,
    eps=1e-8,
)

criterion = nn.CrossEntropyLoss()

print("LoRA parameters:", sum(p.numel() for p in lora_params))
print("Head parameters:", sum(p.numel() for p in head_params))

LoRA parameters: 491520
Head parameters: 2497282


In [39]:
model_param_ids = {id(p) for p in model.module.parameters()}
optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

assert optimizer_param_ids.issubset(model_param_ids), \
    "Optimizer contains parameters outside the current model"

print("Optimizer-model parameter check passed!")

Optimizer-model parameter check passed!


In [40]:

import torch
import torch.nn as nn
from sklearn.metrics import f1_score

EPOCHS = 3

# scaler = torch.amp.GradScaler(
#     "cuda",
#     init_scale=1024.0,
#     growth_interval=2000,
#     enabled=(device.type == "cuda"),
# )

best_val_f1 = -1.0

for epoch in range(EPOCHS):
    train_metrics = train_one_epoch(
    model, train_loader, optimizer, criterion, device
)

    val_metrics = evaluate(
    model, val_loader, criterion, device
)

    val_f1 = f1_score(
        val_metrics["labels"],
        val_metrics["predictions"],
        average="macro",
        zero_division=0,
    )

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}\n"
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['accuracy']:.4f}\n"
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

    base = (
        model.module
        if isinstance(model, nn.DataParallel)
        else model
    )

    # Save each epoch separately
    epoch_path = (
        f"/kaggle/working/best_hierarchical_lora_cjp_{epoch + 1}.pt"
    )
    torch.save(base.state_dict(), epoch_path)

    # Save the best model based on validation macro-F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1

        torch.save(
            base.state_dict(),
            "/kaggle/working/best_hierarchical_lora_cjp.pt",
        )

        print(f"Saved best checkpoint. Macro-F1: {best_val_f1:.4f}")


Training:   0%|          | 0/16153 [00:00<?, ?it/s]

Validation:   0%|          | 0/1256 [00:00<?, ?it/s]


Epoch 1/3
Train Loss: 0.8265 | Train Acc: 0.7588
Val Loss: 1.1041 | Val Acc: 0.7340 | Val Macro-F1: 0.7226
Saved best checkpoint. Macro-F1: 0.7226


Training:   0%|          | 0/16153 [00:00<?, ?it/s]

Validation:   0%|          | 0/1256 [00:00<?, ?it/s]


Epoch 2/3
Train Loss: 0.8234 | Train Acc: 0.7697
Val Loss: 0.8410 | Val Acc: 0.7527 | Val Macro-F1: 0.7503
Saved best checkpoint. Macro-F1: 0.7503


Training:   0%|          | 0/16153 [00:00<?, ?it/s]

Validation:   0%|          | 0/1256 [00:00<?, ?it/s]


Epoch 3/3
Train Loss: 0.8342 | Train Acc: 0.7762
Val Loss: 0.9480 | Val Acc: 0.7523 | Val Macro-F1: 0.7513
Saved best checkpoint. Macro-F1: 0.7513


In [41]:
model.train()
optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
chunk_mask = batch["chunk_mask"].to(device)
labels = batch["labels"].to(device, dtype=torch.long)

# No autocast; no GradScaler for this diagnostic
logits = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    chunk_mask=chunk_mask,
)

loss = criterion(logits.float(), labels)

print("Logits finite:", torch.isfinite(logits).all().item())
print("Loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

loss.backward()

bad_grads = [
    name for name, p in model.named_parameters()
    if p.grad is not None and not torch.isfinite(p.grad).all()
]

print("Non-finite gradients:", bad_grads[:20])

Logits finite: True
Loss: 3.3195853233337402
Loss finite: True
Non-finite gradients: []


In [42]:
base = model.module if isinstance(model, torch.nn.DataParallel) else model

for name, p in base.named_parameters():
    if "v_proj.A" in name or "classifier.4.weight" in name:
        print(
            name,
            "requires_grad =", p.requires_grad,
            "dtype =", p.dtype,
            "finite =", torch.isfinite(p).all().item(),
        )

chunk_encoder.model.blocks.0.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.1.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.2.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.3.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.4.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.5.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.6.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.7.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.8.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True
chunk_encoder.model.blocks.9.attn.v_proj.A requires_grad = True dtype = torch.float32 finite = True


In [43]:
test_dataset = CJPDataset(
    json_path=test_file_paths["Phase_4_CJPE/Test/test.json"],
    converter=converter,
    max_chunk_len=1024,
    overlap=64,
    max_chunks=16,
    has_labels=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,  # start small for GPU memory safety
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_cjp,
)

print("Test examples:", len(test_dataset))
print("Test batches:", len(test_loader))

Test examples: 1517
Test batches: 759


In [44]:
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU total: {total / 1024**3:.2f} GB")
    print(f"GPU free:  {free / 1024**3:.2f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU total: 14.56 GB
GPU free:  3.47 GB
Allocated: 1.08 GB
Reserved:  10.49 GB


In [45]:
import json
import torch
import torch.nn as nn
from tqdm.auto import tqdm

In [46]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from collections import Counter
import json

# Unwrap DataParallel if necessary
if isinstance(model, nn.DataParallel):
    model = model.module

model = model.to(device)
model.eval()

THRESHOLD = 0.5
predictions = []

with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Generating predictions"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        chunk_mask = batch["chunk_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            chunk_mask=chunk_mask,
        )

        # Probability of class 1
        probs = torch.softmax(logits, dim=-1)
        p_class1 = probs[:, 1]

        # Apply the selected validation threshold
        preds = (p_class1 >= THRESHOLD).long().cpu().tolist()

        for doc_id, pred in zip(batch["ids"], preds):
            predictions.append({
                "id": doc_id,
                "pred": int(pred),
            })

print("Total predictions:", len(predictions))
print("Prediction distribution:", Counter(p["pred"] for p in predictions))
print("Sample predictions:", predictions[:5])

# Validate and save
assert len(predictions) == len(test_dataset), "Prediction count mismatch!"
assert len({p["id"] for p in predictions}) == len(predictions), "Duplicate IDs!"
assert all(p["pred"] in (0, 1) for p in predictions), "Invalid labels!"

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print("Saved submission.json")

Generating predictions:   0%|          | 0/759 [00:00<?, ?it/s]

Total predictions: 1517
Prediction distribution: Counter({0: 839, 1: 678})
Sample predictions: [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}, {'id': '1991_137', 'pred': 0}, {'id': '1957_125', 'pred': 1}]
Saved submission.json


In [47]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print("Saved:", PREDICTION_PATH)
print("Documents:", len(predictions))


Saved: /kaggle/working/cs781_phase2/prediction.json
Documents: 1517


In [48]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)
import os
import shutil
from IPython.display import display, FileLink

# Copy to Kaggle's working directory with a simple filename
download_path = "/kaggle/working/submission.zip"

shutil.copy2(SUBMISSION_ZIP, download_path)

print("Exists:", os.path.isfile(download_path))
print("Size:", os.path.getsize(download_path), "bytes")

display(FileLink("submission.zip"))

Codabench submission created:
/kaggle/working/cs781_phase2/submission.zip
Exists: True
Size: 6619 bytes


/kaggle/working/submission.zip

In [49]:
import torch

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        chunk_mask = batch["chunk_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            chunk_mask=chunk_mask,
        )

        probs = torch.softmax(logits, dim=-1)
        all_probs.extend(probs[:, 1].cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

In [50]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

y_true = np.array(all_labels)
p_class1 = np.array(all_probs)

results = []

for threshold in np.arange(0.10, 0.71, 0.05):
    y_pred = (p_class1 >= threshold).astype(int)

    results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy_score(y_true, y_pred),
        "class1_precision": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "class1_recall": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
    })

results = sorted(results, key=lambda x: x["macro_f1"], reverse=True)

for row in results:
    print(row)

{'threshold': 0.1, 'accuracy': 0.7578653922739944, 'class1_precision': 0.7723849372384938, 'class1_recall': 0.7331215250198571, 'macro_f1': 0.757740557091607}
{'threshold': 0.15, 'accuracy': 0.7562724014336918, 'class1_precision': 0.7772065124250214, 'class1_recall': 0.7204130262112788, 'macro_f1': 0.7559927950540943}
{'threshold': 0.4, 'accuracy': 0.7554759060135404, 'class1_precision': 0.7897574123989218, 'class1_recall': 0.6981731532962668, 'macro_f1': 0.7547242992141017}
{'threshold': 0.2, 'accuracy': 0.7550776583034647, 'class1_precision': 0.7795138888888888, 'class1_recall': 0.7132644956314536, 'macro_f1': 0.7546885913709999}
{'threshold': 0.35, 'accuracy': 0.7550776583034647, 'class1_precision': 0.7869875222816399, 'class1_recall': 0.7013502779984114, 'macro_f1': 0.7544194136942706}
{'threshold': 0.3, 'accuracy': 0.7542811628833134, 'class1_precision': 0.7835689045936396, 'class1_recall': 0.704527402700556, 'macro_f1': 0.7537186913868856}
{'threshold': 0.25, 'accuracy': 0.753484